# Workflow development

This is a little playground for me to develop functions that will be used in the final workflow.

## Imports

In [33]:
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem import Draw, PandasTools
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
import pandas as pd
import numpy as np
import MDAnalysis as mda
import nglview as nv
import os
from openmm.app import PDBFile
from pdbfixer import PDBFixer
from vina import Vina
import useful_rdkit_utils as uru
from molscrub import Scrub
from tqdm.auto import tqdm
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate

tqdm.pandas()
RDLogger.DisableLog('rdApp.*') 

## Protein preparation

**NOTE TO SELF**: Should look into replacing `os.system` with `os.subprocess` - apparently that's safer and more professional.

In [19]:
def _fix_protein(input_protein_path:str, output_directory_path:str):
    """Runs PDBFixer on the input protein structure to fix any issues such as missing residues, missing atoms, and nonstandard residues.
    The fixed structure is then saved to an internally specified output path (fixed_{input_protein_path}).

    Args:
        input_protein_path (str): The path to the input protein PDB file
        output_directory_path (str): The path to the directory where the fixed protein PDB file will be saved

    Returns:
        str: The path where the fixed protein PDB file has been saved.
    """
    # Initialise PDBFixer with input protein structure
    fixer = PDBFixer(filename=input_protein_path)

    # Identify problems with the structure and fix them
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.findNonstandardResidues()
    fixer.removeHeterogens(keepWater=False) # remove anything that isn't protein, including water and ligand(s).

    # Fix identified problems
    fixer.addMissingAtoms()
    fixer.replaceNonstandardResidues()

    # Save the fixed structure to the following output path
    output_filename = "fixed_"+os.path.split(input_protein_path)[-1] # grabs current name of input file and prepends "fixed_"
    fixed_protein_output_path = os.path.join(output_directory_path, output_filename)
    with open(fixed_protein_output_path, 'w') as f:
        # Toplology, Positions, file stream, and keep chain ID's
        PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
    
    return fixed_protein_output_path

def _protonate_protein(input_protein_path:str, output_directory_path:str, pH=7.4):
    """Runs pdb2pqr on the input protein structure to add hydrogens and assign protonated states based on the specified pH.
    The protonated structure is then saved to an internally specified output path.

    Args:
        input_protein_path (str): The path to the input protein PDB file
        output_directory_path (str): The path to the directory where the protonated protein .pqr and .pdb files will be saved
        pH (float, optional): The pH at which to assign protonated states. Defaults to 7.4 (physiological pH).

    Returns:
        str: The path where the protonated protein .pqr file has been saved.
    """

    # Defining output path and creating protonated files. A protonated .pdb file is created primarily for future visualisation.
    output_filename = os.path.split(input_protein_path)[-1].split(".")[0]+"_protonated" # grabs current name of input file, removes file extension, and appends "_protonated"
    protonated_protein_output_path = os.path.join(output_directory_path, output_filename)
    os.system(f"pdb2pqr --pdb-output={protonated_protein_output_path}.pdb --pH={pH} {input_protein_path} {protonated_protein_output_path}.pqr --whitespace")

    return f"{protonated_protein_output_path}.pqr"

def _pdb_to_pdbqt(input_protein_pqr_path:str, output_directory_path:str):
    """Writes a .pdbqt file ready for Autodock Vina given a .pqr generated by pdb2pqr.

    Args:
        input_protein_pqr_path (str): The path to the input protein .pqr file (created by pdb2pqr).
        output_directory_path (str): The path to the directory where the .pdbqt file will be saved.
    
    Returns:
    str: The path where the protein .pdbqt file has been saved.
    """

    u = mda.Universe(input_protein_pqr_path)
    output_filename = os.path.split(input_protein_pqr_path)[-1].split(".")[0]+".pdbqt" # grabs current name of input file, removes file extension, and appends ".pdbqt"
    output_filename = output_filename.replace("_protonated", "").replace("fixed", "") # removes "_protonated" and "fixed_" from the filename to match the original protein name
    pdbqt_output_path = os.path.join(output_directory_path, output_filename)
    u.atoms.write(pdbqt_output_path)

    # Read in the just-written PDBQT file, replace text, and write back
    with open(pdbqt_output_path, 'r') as file:
        file_content = file.read()

    # Replace 'TITLE' and 'CRYST1' with 'REMARK'
    file_content = file_content.replace('TITLE', 'REMARK').replace('CRYST1', 'REMARK')

    # Write the modified content back to the file
    with open(pdbqt_output_path, 'w') as file:
        file.write(file_content)
    
    return pdbqt_output_path

# Final function which will prepare the protein structure files with one function call.
def prepare_protein(input_protein_path:str, output_directory_path:str, pH=7.4):
    """Convenience function which fixes a protein structure with PDBFixer if any issues are present, then protonates the protein with
    pdb2pqr (propka under the hood) at a given pH, and converts the protonated structure file (.pqr used in this case) to a .pdbqt file
    ready for docking with Autodock Vina.

    Args:
        input_protein_path (str): Input protein PDB filepath.
        output_directory_path (str): Output directory path for the .pdbqt file.
        pH (float, optional): The pH at which to assign protonated states. Defaults to 7.4 (physiological pH).

    Returns:
    str: The path where the prepared protein .pdbqt file has been saved.
    """

    fixed_protein_path = _fix_protein(input_protein_path, output_directory_path)
    protonated_protein_pqr_path = _protonate_protein(fixed_protein_path, output_directory_path, pH=pH)
    pdbqt_path = _pdb_to_pdbqt(protonated_protein_pqr_path, output_directory_path)

    return pdbqt_path


## Ligand preparation

In [20]:
def _protonate_ligand(smiles:str, pH=7.4, pH_deviation=2) -> list[str]:
    """Protonate a ligand molecule given its SMILES string, using the molscrub Scrub class to assign protonation states and
    enumerate tautomers based on the specified pH and deviation range.

    Args:
        smiles (str): Ligand SMILES string.
        pH (float, optional): pH at which to protonate the ligand. Defaults to 7.4.
        pH_deviation (int, optional): Deviation from the specified pH to consider. Defaults to 2.

    Returns:
        list[str]: A list of protonated ligand SMILES strings, including tautomers.
    """

    scrub = Scrub(
    ph_low=pH - pH_deviation,
    ph_high=pH + pH_deviation,
    skip_gen3d=True # We can skip 3D generation here as we will generate 3D conformers later in the workflow after docking (using Auto3D)
    )

    mol = Chem.MolFromSmiles(smiles)
    scrubbed_mols = scrub(mol)
    protonated_smiles = [Chem.MolToSmiles(mol) for mol in scrubbed_mols]

    return protonated_smiles

# Taken and slightly edited from Auto3D stereochemistry documentation: https://auto3d.readthedocs.io/en/latest/example/stereochemistry.html
def _enumerate_stereoisomers(smiles:str, max_isomers=16, only_unassigned=True) -> list[str]:
    """Enumerate all stereoisomers of a molecule. By default, if stereochemistry is specified, it will not be enumerated
    (controlled by the only_unassigned parameter). This function treats both E/Z and R/S stereoisomerism. Generated
    stereoisomers will be checked to confirm if they can be converted into a 3D structure, and if not, it is assumed the
    molecule is nonsensical and the function will return NaN.

    Args:
        smiles (str): Input SMILES (can have undefined stereochemistry).
        max_isomers (int, optional): Maximum number of isomers to generate. Defaults to 16.
        only_unassigned (bool, optional): If set (default), stereocenters which have specified stereochemistry will not be perturbed unless they are part of a relative stereo group. Defaults to True.

    Returns:
        list[str]: List of enumerated SMILES with stereochemical information.
    """

    mol = Chem.MolFromSmiles(smiles)
    
    # Configure enumeration options
    opts = StereoEnumerationOptions(
        tryEmbedding=True,  # Verify 3D embedding is possible. If not, this function returns NaN.
        unique=True,        # Remove duplicates
        maxIsomers=max_isomers,
        onlyUnassigned=only_unassigned
    )
    
    isomers = list(EnumerateStereoisomers(mol, options=opts))
    isomer_smiles = [Chem.MolToSmiles(iso, isomericSmiles=True, canonical=True) for iso in isomers]

    return isomer_smiles

def _generate_3d_molecule(smiles:str, max_iter=1000) -> Chem.rdchem.Mol:
    # An attempt at generating 3D conformers of a molecule for docking without relying on auto3d.
    """Generate a 3D conformer of a given SMILES string using RDKit. The structure is then optimised using AllChem.MMFFOptimizeMolecule.

    Args:
        smiles (str): SMILES string.
        max_iter (int, optional): Maximum number of iterations to optimise a 3D structure. Defaults to 1000.

    Returns:
        Chem.rdchem.Mol: The 3D structure for the given SMILES.
    """
    mol = Chem.MolFromSmiles(smiles)
    mol_with_H = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol_with_H) # Embed molecule is done inplace
    AllChem.MMFFOptimizeMolecule(mol_with_H, maxIters=max_iter) # returns optimisation status - 0=converged, 1=not converged

    return mol_with_H

def _write_3d_mols_to_sdf(mol_3d_list:list[Chem.rdchem.Mol], smiles_list:list[str], sdf_path:str):
    """Write 3D molecule structures to an .sdf file for later analysis.

    Args:
        mol_3d_list (list[Chem.rdchem.Mol]): List of 3D RDKit Mol structures, as made with _generate_3d_molecule.
        smiles_list (list[str]): List of SMILES used to generate the 3D structures to save with each structure in the .sdf file.
        sdf_path (str): Output filepath for .sdf file.
    """
    writer = Chem.SDWriter(sdf_path)

    for mol, smi in zip(mol_3d_list, smiles_list):
        # Store the SMILES as an SDF property
        mol.SetProp("_SMILES", smi)
        writer.write(mol)
    
    writer.close()

def _write_ligand_pdbqt(mol_3d:Chem.rdchem.Mol, pdbqt_path:str):
    """Write a .pdbqt file from a 3D RDKit Mol object, as generated by _generate_3d_molecule.

    Args:
        mol_3d (Chem.rdchem.Mol): 3D RDKit Mol object, as generated by _generate_3d_molecule.
        pdbqt_path (str): Output filepath for .pdbqt file.
    """
    meeko_prep = MoleculePreparation()
    mol_setup = meeko_prep.prepare(mol_3d)[0] # there will only ever be one molecule setup in this basic workflow
    pdbqt_string = PDBQTWriterLegacy.write_string(mol_setup)[0] # write_string returns a tuple - the pdbqt string and an error message (if relevant, otherwise it's blank)

    # file writing
    with open(pdbqt_path, "w") as f:
        f.write(pdbqt_string)

def _write_multiple_ligand_pdbqt_files(mol_3d_list:list[Chem.rdchem.Mol], pdbqt_output_directory:str) -> str:
    """A version of _write_ligand_pdbqt which applies to a list of 3D RDKit Mol objects to write .pdbqt files
    for each molecule in the list. This function returns a list of .pdbqt filepaths for each molecule passed in.

    Args:
        mol_3d_list (list[Chem.rdchem.Mol]): 3D RDKit Mol objects, as generated by _generate_3d_molecule.
        pdbqt_output_directory (str): Output directory for .pdbqt files.

    Returns:
        list[str]: List of .pdbqt filepaths for each molecule passed in.
    """
    ligand_pdbqt_dir = os.path.join(pdbqt_output_directory, "ligand_pdbqts")
    os.makedirs(ligand_pdbqt_dir)
    ligand_pdbqt_filepath_list = []
    for i, mol in enumerate(tqdm(mol_3d_list)):
        ligand_pdbqt_filepath = os.path.join(ligand_pdbqt_dir, f"{i}.pdbqt")
        ligand_pdbqt_filepath_list.append(ligand_pdbqt_filepath)
        _write_ligand_pdbqt(mol, ligand_pdbqt_filepath)
    return ligand_pdbqt_filepath_list

def prepare_ligands(smiles_list:list[str], output_path:str, pH=7.4, pH_deviation=2, enumerate_only_unassigned=True, max_iter=1000) -> list[str]:
    """Convenience function which generates the protonation states and tautomers of a list of SMILES, then enumerates
    stereoisomers, generating 3D structures for each stereoisomer and optimising the structures, and saving .pdbqt files
    for docking with Autodock Vina.

    Args:
        smiles_list (list[str]): List of SMILES strings.
        output_path (str): Directory to write all files to.
        pH (float, optional): pH to protonate ligands. Defaults to 7.4.
        pH_deviation (int, optional): Deviation from the specified pH to consider for ligand protonation. Defaults to 2.
        enumerate_only_unassigned (bool, optional): If set (default), stereocenters which have specified stereochemistry will not be perturbed unless they are part of a relative stereo group. Defaults to True.
        max_iter (int, optional): Maximum number of iterations to optimise a 3D structure. Defaults to 1000.
    
    Returns:
        list[str]: List of absolute filepaths for each generated ligand .pdbqt file.
    """
    
    # For below: progress_apply is from tqdm.pandas() and applies a function whilst showing a progress bar
    ligand_df = pd.DataFrame({"annalog": smiles_list})

    print("Protonating molecules...")
    ligand_df["protonated"] = ligand_df["annalog"].progress_apply(_protonate_ligand, pH=pH, pH_deviation=pH_deviation)
    ligand_df = ligand_df.explode("protonated", ignore_index=True)

    print("Enumerating isomers...")
    ligand_df["isomers"] = ligand_df["protonated"].progress_apply(_enumerate_stereoisomers, only_unassigned=enumerate_only_unassigned)
    ligand_df = ligand_df.explode("isomers", ignore_index=True)
    invalid_smiles = ligand_df.isna().sum().sum() # when generating stereoisomers, NaN can be returned if a SMILES cannot be converted to 3D, thus it is considered nonsensical.
    print(f"{invalid_smiles} invalid SMILES detected.")
    if(invalid_smiles > 0):
        ligand_df = ligand_df.dropna()
        print("Invalid SMILES removed.")

    print("Deduplicating isomers...")
    len_before_deduplication = ligand_df.shape[0]
    ligand_df = ligand_df.drop_duplicates("isomers", ignore_index=True)
    len_after_deduplication = ligand_df.shape[0]
    print(f"{len_before_deduplication} molecules identified after enumeration.", end=" ")
    print(f"{len_before_deduplication-len_after_deduplication} entries removed.")

    ligand_info_path = os.path.join(output_path, "prepped_ligands.csv")
    print(f"Saving enumerated ligand smiles to {ligand_info_path}")
    ligand_df.to_csv(ligand_info_path)

    print("Generating 3D structures per SMILES...") # this is done more to check the structures later on, if the user wishes
    mols_3d = ligand_df["isomers"].progress_apply(_generate_3d_molecule, max_iter=max_iter).to_list()
    sdf_path = os.path.join(output_path, "prepped_ligands.sdf")
    print(f"Saving 3D structures to {sdf_path}")
    _write_3d_mols_to_sdf(mols_3d, ligand_df["isomers"].to_list(), sdf_path)
    
    print("Writing .pdbqt files of each ligand...")
    ligand_pdbqt_filepath_list = _write_multiple_ligand_pdbqt_files(mols_3d, output_path)
    print(f"{len(ligand_pdbqt_filepath_list)} molecules have been prepped for docking.")
    
    return ligand_pdbqt_filepath_list

## ANNalog generation setup

In [21]:
def annalog_generate(n_mol_to_gen:int, input_path_or_smiles:str, output_file:str, method="beam", number_of_variants=5):
    """Generate SMILES with ANNalog. Variants are used to generate multiple variant SMILES per input SMILES.
    The total number of molecules to generate is passed in, and internally this is converted into a number of molecules
    to generate per input SMILES, additionally given the number of variants to generate per input SMILES.

    Args:
        n_mol_to_gen (int): Number of molecules to generate in total.
        input_path_or_smiles (str): Path to input file or SMILES string
        output_file (str): Path to output file, as a .tsv or .csv file.
        method (str, optional): Generation method. Defaults to "beam". Other options are: "BF-beam" and "sampling".
        number_of_variants (int, optional): Number of variants to generate. Defaults to 5.
    """
    # computing the number of molecules to generate per input SMILES, given the total number of molecules to generate.
    n_mol_to_gen_per_input = n_mol_to_gen // number_of_variants # integer division to ensure we get an integer number of molecules to generate per input SMILES.
    os.system(f"annalog-generate -i '{input_path_or_smiles}' -n {n_mol_to_gen_per_input} -m {method} -o {output_file} -e variants --variant-number {number_of_variants}")

def load_generated_molecules(generated_molecules_path:str):
    """Convenience function to load generated molecules from a .tsv file and add RDKit molecule columns for both input and generated SMILES.

    Args:
        generated_molecules_path (str): Path to the .tsv file containing generated molecules.

    Returns:
        pd.DataFrame: DataFrame containing the generated molecules with RDKit molecule columns.
    """

    df = pd.read_csv(generated_molecules_path, sep="\t")
    return df

def deduplicate_generated_molecules(generated_df:pd.DataFrame) -> pd.DataFrame:
    """Deduplicates a DataFrame created from ANNalog's generation output. Duplicated generated molecules are removed, along with generated
    molecules that are identical to an input molecule.

    Args:
        generated_df (pd.DataFrame): DataFrame based on ANNalog generation output.

    Returns:
        pd.DataFrame: Deduplicated ANNalog generation results DataFrame.
    """

    generated_df["canonical_input_smiles"] = generated_df["input_smiles"].apply(Chem.CanonSmiles)
    generated_df["canonical_generated_smiles"] = generated_df["generated_smiles"].apply(Chem.CanonSmiles)
    generated_df = generated_df.drop_duplicates("canonical_generated_smiles") # remove duplicate generated molecules
    generated_df = generated_df[~generated_df["canonical_generated_smiles"].isin(generated_df["canonical_input_smiles"])] # remove generated molecules that are the same as its input molecule
    generated_df = generated_df.reset_index(drop=True)
    return generated_df


## Docking functions

### Setup

In [34]:
def _get_receptor_grid(protein_with_ligand_struct_path:str, buffer=5):
    """Calculate a docking receptor grid's box centre and box dimensions based on a protein-ligand co-crystal.
    The box centre is calculated as the centroid of the bound ligand, and the box dimensions are calculated as
    the maximum extent of the ligand in each dimension, plus a buffer to ensure the ligand fits comfortably within
    the box.

    Args:
        protein_with_ligand_struct_path (str): Filepath to a protein-ligand co-crystal structure (PDB format).
        buffer (int, optional): Buffer size in Angstroms to add to the box dimensions. Defaults to 5.

    Returns:
        tuple[list]: A tuple containing the box centre and box dimensions.
    """
    u = mda.Universe(protein_with_ligand_struct_path)
    ligand = u.select_atoms("not protein")
    box_centre = ligand.center_of_geometry()
    box_centre = box_centre.tolist()
    box_dimensions = np.ceil(ligand.positions.max(axis=0) - ligand.positions.min(axis=0)) + buffer # np.ceil is used to round up the dimensions to the larger nearest integer for ease of reporting size.
    box_dimensions = box_dimensions.tolist()

    return box_centre, box_dimensions

def vina_setup_receptor(vina_object:Vina, protein_with_ligand_struct_path:str, protein_pdbqt_path:str) -> Vina:
    """Initialise and setup a Vina object with a receptor structure and receptor grid.

    Args:
        vina_object (Vina): An instance of the Vina class from the vina package.
        protein_with_ligand_struct_path (str): Filepath to a protein-ligand co-crystal structure (PDB format) to setup receptor grid.
        protein_pdbqt_path (str): Filepath to the prepared protein .pdbqt file.

    Returns:
        Vina: The initialised Vina object with the receptor structure and receptor grid set up.
    """
    box_centre, box_dimensions = _get_receptor_grid(protein_with_ligand_struct_path)
    vina_object.set_receptor(protein_pdbqt_path)
    vina_object.compute_vina_maps(center=box_centre, box_size=box_dimensions)
    return vina_object

def _vina_dock_ligand(vina_object:Vina, ligand_pdbqt:str, n_poses:int, exhaustiveness:int) -> Vina:
    """Dock a ligand to a receptor using a Vina object already set up with vina_setup_receptor.

    Args:
        vina_object (Vina): An instance of the Vina class from the vina package, with the receptor structure and grid set up.
        ligand_pdbqt (str): Filepath to the prepared ligand .pdbqt file.
        n_poses (int): Number of docking poses to generate.
        exhaustiveness (int): Exhaustiveness of the global search. Higher values lead to more thorough searches but take longer.

    Returns:
        list[tuple]: A list of tuples containing the docking poses and their corresponding scores.
    """
    vina_object.set_ligand_from_file(ligand_pdbqt)
    vina_object.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
    return vina_object

### Pose and energy collection

In [35]:
def _get_best_docking_energy(vina_object:Vina) -> float:
    """Returns the docking score (energy) of the best docked pose from Autodock Vina.

    Args:
        vina_object (Vina): Vina object after docking has been performed to take the energy from.

    Returns:
        float: The best docked pose energy.
    """
    best_energy = vina_object.energies()[0,0] # this grabs the total energy of the best (lowest energy) pose
    return best_energy

def _save_docking_energies(smiles_df_path:str, output_path:str, docking_energy_list:list[float]):
    """Given a list of energies as acquired through get_best_docking_energy, save the results to the (likely .csv) file
    generated earlier in the workflow that contains the SMILES strings of the different isomers generated from the
    original ANNalog SMILES.

    Args:
        smiles_df_path (str): Path to file containing SMILES used to generate 3D molecule structures used for docking.
        output_path (str): Path to save the modified file with docking scores/energies.
        docking_energy_list (list[float]): List of docking score energies per molecule docked.
    """
    smiles_df = pd.read_csv(smiles_df_path, index_col=0)
    smiles_df["docking_energy"] = docking_energy_list
    smiles_df.to_csv(output_path)

def _get_best_docking_pose(vina_object:Vina) -> str:
    """Returns the best docked pose for a molecule from Autodock Vina.

    Args:
        vina_object (Vina): Vina object after docking has been performed to take the pose from.

    Returns:
        str: PDBQT file string of the docked pose.
    """
    output_pdbqt = vina_object.poses(n_poses=1)
    return output_pdbqt

def _save_docking_poses(pose_pdbqt_list:list[str], sdf_output_path:str):
    """Save the docking poses collected from each molecule into one .sdf file.

    Args:
        pose_pdbqt_list (list[str]): List of PDBQT strings as generated by get_best_docking_pose.
        smiles_list (list[str]): List of SMILES used to generated the molecules that were docked, used to assign bond orders before saving.
        sdf_output_path (str): Filepath to save .sdf file to.
    """
    f = Chem.SDWriter(sdf_output_path)
    for pdbqt in pose_pdbqt_list:
        pmol = PDBQTMolecule(pdbqt, skip_typing=True)
        rdmol_list = RDKitMolCreate.from_pdbqt_mol(pmol)
        best_pose = rdmol_list[0] # there should only be one pose saved. Otherwise one should iterate over pmol to save each pose.
        f.write(best_pose)
    f.close()

### Full docking protocol function

In [36]:
def _dock_and_score(ligand_pdbqt_filepath_list:list[str], prepared_vina_object:Vina, n_poses:int, exhaustiveness:int) -> tuple[list[float], list[str]]:
    """Given a list of prepared ligand PDBQT filepaths, dock to a receptor as defined in a prepared Vina object and return
    a list of docking scores (energies) and pose PDBQT strings per ligand.

    Args:
        ligand_pdbqt_filepath_list (list[str]): List of prepared ligand PDBQT filepaths.
        prepared_vina_object (Vina): A set up Vina object that already has a receptor and receptor grid set up, as done with vina_setup_receptor.
        n_poses (int): Vina parameter - Number of poses to generate for docking. This controls how many poses are returned after docking.
        exhaustiveness (int): Vina parameter - Exhaustiveness of the global search. Higher values lead to more thorough searches but take longer.

    Returns:
        tuple[list[float], list[str]]: A tuple of, 1. a list of docking scores per ligand, and 2. a list of pose PDBQT strings per ligand.
    """
    
    docking_energy_list = []
    docked_pose_pdbqt_list = []
    for ligand in tqdm(ligand_pdbqt_filepath_list):
        v = _vina_dock_ligand(prepared_vina_object, ligand, n_poses=n_poses, exhaustiveness=exhaustiveness)
        docking_energy_list.append(_get_best_docking_energy(v))
        docked_pose_pdbqt_list.append(_get_best_docking_pose(v))
    
    return docking_energy_list, docked_pose_pdbqt_list

def _save_docking_results(smiles_df_path:str, output_directory:str, docking_energy_list:list[float], docking_pose_pdbqt_list:list[str]):
    """Save docking poses as a multi-ligand .sdf and energies as a .csv file associated with ligand SMILES strings.

    Args:
        smiles_df_path (str): Filepath to file containing enumerated isomer SMILES made earlier in the workflow.
        output_directory (str): Directory to save docking results.
        docking_energy_list (list[float]): List of docking scores as generated with dock_and_score.
        docking_pose_pdbqt_list (list[str]): List of docked pose PDBQT strings as generated with dock_and_score.
    """
    
    smiles_and_energies_output_path = os.path.join(output_directory, "smiles_and_energies.csv")
    pose_sdf_path = os.path.join(output_directory, "docked_poses.sdf")
    _save_docking_energies(smiles_df_path, smiles_and_energies_output_path, docking_energy_list)
    print(f"Energies saved to {smiles_and_energies_output_path}")
    _save_docking_poses(docking_pose_pdbqt_list, pose_sdf_path)
    print(f"Poses saved to {pose_sdf_path}")

def docking(ligand_pdbqt_filepath_list:list[str], prepared_vina_object:Vina, n_poses:int, exhaustiveness:int, smiles_df_path:str, output_directory:str):
    """Convenience function to perform docking and save docking results.

    Args:
        ligand_pdbqt_filepath_list (list[str]): List of prepared ligand PDBQT filepaths.
        prepared_vina_object (Vina): A set up Vina object that already has a receptor and receptor grid set up, as done with vina_setup_receptor.
        n_poses (int): Vina parameter - Number of poses to generate for docking. This controls how many poses are returned after docking.
        exhaustiveness (int): Vina parameter - Exhaustiveness of the global search. Higher values lead to more thorough searches but take longer.
        smiles_df_path (str): Filepath to file containing enumerated isomer SMILES made earlier in the workflow.
        output_directory (str): Directory to save docking results.
    """
    docking_energy_list, docked_pose_pdbqt_list = _dock_and_score(ligand_pdbqt_filepath_list, prepared_vina_object, n_poses, exhaustiveness)
    _save_docking_results(smiles_df_path, output_directory, docking_energy_list, docked_pose_pdbqt_list)

## Results collection

### Clustering

### Physicochemical property collection

### Plotting

# Workflow

In [12]:
# workflow settings
workflow_rounds = 5 # run the workflow five times
## preparation settings (protein+ligand)
initial_ligand_smiles = "Cc1c(n(nc1C(=O)NC23CC4CC(C2)CC(C4)C3)CCCCCO)c5ccccc5" # PDB ligand ID: 9JU, bound to PDB ID 5ZTY
pH = 7.4 # physiological pH - specified for ligand and protein protonation
pH_deviation = 2 # deviation of +/- 2 log units for ligand protonation
enumerate_only_unassigned = True # only enumerate undefined stereocentres.
max_opt_iter = 1000 # when generating 3D molecules, this is the maximum number of optimisation steps for the 3D geometry optimisation

## docking settings
exhaustiveness = 16 # doubled from the default of 8
n_poses = 1 # only save the best pose per ligand.


# generation settings
num_of_mols_to_gen = 100
gen_algo = "beam"

# input and output directories for the workflow
workflow_results_dir = os.path.join("results", "workflow") # working directory for the workflow results, which will contain subdirectories for each round of generation and docking.
input_protein_structure_path = os.path.join("Structures", "refined_cnr2_human_5ZTY_inactive.pdb") # protein structure to dock to (refined version of PDB ID: 5ZTY from GPCRdb - CB2 receptor)
output_protein_structures_path = os.path.join(workflow_results_dir, "protein") # output directory path for the prepared protein files
run_directory = os.path.join(workflow_results_dir, gen_algo) # naming the directory to save results to after the generation algorithm used

# protein preparation
print("Preparing protein structure for docking...")
os.makedirs(output_protein_structures_path)
protein_pdbqt_path = prepare_protein(input_protein_structure_path, output_protein_structures_path, pH=pH)

v=Vina(sf_name="vina", verbosity=0) # using the default vina scoring function, and setting the verbosity to 0 to silence Vina output
v=vina_setup_receptor(v, input_protein_structure_path, protein_pdbqt_path)

Preparing protein structure for docking...


INFO:PDB2PQR v3.7.1: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: results/workflow/protein/fixed_refined_cnr2_human_5ZTY_inactive.pdb
INFO:Setting up molecule.
INFO:Created biomolecule object with 360 residues and 2785 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:This biomolecule is clean.  No repair needed.
INFO:Updating disulfide bridges.
INFO:Debumping biomolecule.
INFO:Adding hydrogens to biomolecule.
INFO:Debumping biomolecule (again).
INFO:Optimizing hydrogen bonds
INFO:Applying fo

In [8]:


round_no = 0 # this is a counter that will be added to as generation and docking rounds are completed

# Defining and creating ligand and docking result directories
generation_round_dir = os.path.join(run_directory, f"round_{round_no}")
ligand_directory = os.path.join(generation_round_dir, "ligand_prep")
os.makedirs(ligand_directory)
docking_results_directory = os.path.join(generation_round_dir, "docking")
os.makedirs(docking_results_directory)
annalog_output_filepath = os.path.join(ligand_directory, "annalog_generated_molecules.tsv")

print(f"Generating {num_of_mols_to_gen} molecules...")
annalog_generate(num_of_mols_to_gen, initial_ligand_smiles, annalog_output_filepath, method=gen_algo)
generated_df = load_generated_molecules(annalog_output_filepath)
print("Deduplicating generated molecules...")
num_of_mols_before_deduplication = generated_df.shape[0]
generated_df = deduplicate_generated_molecules(generated_df)
generated_smiles = generated_df["generated_smiles"].to_list()
print(f"{len(generated_smiles)}/{num_of_mols_before_deduplication} molecules remain after deduplication.")

print("Preparing ligands for docking...")
ligand_pdbqt_filepaths = prepare_ligands(generated_smiles,
                                         ligand_directory,
                                         pH=pH,
                                         pH_deviation=pH_deviation,
                                         enumerate_only_unassigned=enumerate_only_unassigned,
                                         max_iter=max_opt_iter)

Generating 100 molecules...


[INFO] input SMILES changed after RDKit normalization, using RDKit processed version : Cc1c(-c2ccccc2)n(CCCCCO)nc1C(=O)NC12CC3CC(C1)CC(C3)C2 as input


Deduplicating generated molecules...
57/100 molecules remain after deduplication.
Preparing ligands for docking...
Protonating molecules...


  0%|          | 0/57 [00:00<?, ?it/s]

Enumerating isomers...


...

0 invalid SMILES detected.
Deduplicating isomers...
73 molecules identified after enumeration. 2 entries removed.
Saving enumerated ligand smiles to results/workflow/beam/round_0/ligand_prep/prepped_ligands.csv
Generating 3D structures per SMILES...


  0%|          | 0/71 [00:00<?, ?it/s]

Saving 3D structures to results/workflow/beam/round_0/ligand_prep/prepped_ligands.sdf
Writing .pdbqt files of each ligand...


  0%|          | 0/71 [00:00<?, ?it/s]

71 molecules have been prepped for docking.


In [37]:
enumerated_smiles_path = os.path.join(ligand_directory, "prepped_ligands.csv")
print("Docking...")
docking(ligand_pdbqt_filepaths, v, n_poses, exhaustiveness, enumerated_smiles_path, docking_results_directory)

Docking...


  0%|          | 0/71 [00:00<?, ?it/s]

Energies saved to results/workflow/beam/round_0/docking/smiles_and_energies.csv
Poses saved to results/workflow/beam/round_0/docking/docked_poses.sdf
